# cadflow sim micro-environment (Colab)

Recreates the VM's user-space solver stack — **CalculiX 2.21** (`ccx`) and **OpenFOAM 1912**
(`simpleFoam`, `blockMesh`) — from extracted .debs under `$HOME/.local`, exactly as it was
laid out in the VM. No `apt`, no root, ~20 s to restore.

**Read cell 1 before planning a big run.** Free Colab gives 2 vCPUs. These solvers are
CPU/MPI-bound and the GPU does nothing for them.

In [ ]:
#@title 1. Resource reality check
import os, multiprocessing, subprocess
print('vCPUs:', multiprocessing.cpu_count())
!free -g | head -2
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || print('no GPU (fine for solvers)')
print()
print('Reference: the source VM host has 24 threads.')
print('A simpleFoam/ccx case here runs roughly an order of magnitude slower.')
print('Colab is one runtime -- it does not fan out. For many cases in parallel, use Modal.')

In [ ]:
#@title 2. Restore the solver stack
# Upload solver-env.tar.gz to Drive once, then this cell is all you need per session.
from google.colab import drive
drive.mount('/content/drive')

TARBALL = '/content/drive/MyDrive/cadflow/solver-env.tar.gz'  #@param {type:"string"}

import os, pathlib, shutil, subprocess
HOME = os.path.expanduser('~')
assert os.path.exists(TARBALL), f'not found: {TARBALL} -- upload solver-env.tar.gz there first'

# The tarball holds  cadflow-solvers/  and  bin/  ; the wrappers hardcode $HOME/.local,
# so lay them out exactly that way.
os.makedirs(f'{HOME}/.local', exist_ok=True)
!tar xzf "$TARBALL" -C /tmp/
!rm -rf $HOME/.local/cadflow-solvers && mv /tmp/cadflow-solvers $HOME/.local/
!mkdir -p $HOME/.local/bin && cp /tmp/bin/* $HOME/.local/bin/ && chmod +x $HOME/.local/bin/*
os.environ['PATH'] = f"{HOME}/.local/bin:" + os.environ['PATH']
print('restored to', f'{HOME}/.local')
!ls -1 $HOME/.local/bin

In [ ]:
#@title 3. Verify the solvers actually run
# ccx with no args prints usage and exits non-zero -- that alone proves the
# dynamic linker resolved SPOOLES/ARPACK/OpenMPI correctly.
!$HOME/.local/bin/ccx 2>&1 | head -5
print('---')
!$HOME/.local/bin/simpleFoam -help 2>&1 | head -8
print('---')
!ldd $HOME/.local/cadflow-solvers/calculix-ccx_2.21-1_amd64/usr/bin/ccx 2>&1 | grep -c 'not found' \
  && echo '^ count of unresolved libs (0 is what you want)'

In [ ]:
#@title 4. Python side (meshing + geometry)
# Only what the sim path needs -- not the full training env.
!pip install -q gmsh trimesh meshio shapely numpy scipy ezdxf
import gmsh, trimesh, meshio
gmsh.initialize(); print('gmsh', gmsh.option.getString('General.Version')); gmsh.finalize()
print('trimesh', trimesh.__version__, '| meshio', meshio.__version__)
# cadquery is heavy (OCP ~1 GB) -- install only if you generate CAD here rather than
# shipping pre-built .step/.msh files in:
# !pip install -q cadquery

In [ ]:
#@title 5. Smoke test -- one-element CalculiX solve
import os, subprocess, pathlib, textwrap
HOME = os.path.expanduser('~')
case = pathlib.Path('/content/smoke'); case.mkdir(exist_ok=True)

(case / 'job.inp').write_text(textwrap.dedent('''\
    *NODE, NSET=Nall
    1, 0.0, 0.0, 0.0
    2, 1.0, 0.0, 0.0
    3, 1.0, 1.0, 0.0
    4, 0.0, 1.0, 0.0
    5, 0.0, 0.0, 1.0
    6, 1.0, 0.0, 1.0
    7, 1.0, 1.0, 1.0
    8, 0.0, 1.0, 1.0
    *ELEMENT, TYPE=C3D8, ELSET=Eall
    1, 1, 2, 3, 4, 5, 6, 7, 8
    *NSET, NSET=Fixed
    1, 2, 3, 4
    *BOUNDARY
    Fixed, 1, 3
    *MATERIAL, NAME=STEEL
    *ELASTIC
    210000.0, 0.3
    *SOLID SECTION, ELSET=Eall, MATERIAL=STEEL
    *STEP
    *STATIC
    *CLOAD
    7, 3, 1000.0
    *NODE FILE
    U
    *EL FILE
    S
    *END STEP
    '''))

r = subprocess.run([f'{HOME}/.local/bin/ccx', 'job'], cwd=case,
                   capture_output=True, text=True, timeout=300)
print(r.stdout[-1500:])
print('stderr:', r.stderr[-500:])
print('rc =', r.returncode)
print('produced:', sorted(p.name for p in case.iterdir()))
assert (case / 'job.frd').exists(), 'no .frd -- solve did not complete'
print('\nSOLVER WORKS')

In [ ]:
#@title 6. Batch pattern -- checkpoint per case, resume after a disconnect
import json, pathlib, subprocess, os, time
HOME = os.path.expanduser('~')
OUT = pathlib.Path('/content/drive/MyDrive/cadflow/results'); OUT.mkdir(parents=True, exist_ok=True)
DONE = OUT / 'completed.json'
done = set(json.loads(DONE.read_text())) if DONE.exists() else set()

def run_case(case_dir, job='job', timeout=3600):
    """Run one ccx case; skip if already done. Results land in Drive as they finish,
    so a runtime kill costs you at most the case in flight."""
    name = pathlib.Path(case_dir).name
    if name in done:
        return 'skipped'
    t0 = time.time()
    r = subprocess.run([f'{HOME}/.local/bin/ccx', job], cwd=case_dir,
                       capture_output=True, text=True, timeout=timeout)
    ok = (pathlib.Path(case_dir) / f'{job}.frd').exists()
    if ok:
        dest = OUT / name; dest.mkdir(exist_ok=True)
        for ext in ('.frd', '.dat', '.sta'):
            f = pathlib.Path(case_dir) / f'{job}{ext}'
            if f.exists(): f.replace(dest / f.name)
        done.add(name); DONE.write_text(json.dumps(sorted(done)))
    print(f'{name}: {"ok" if ok else "FAILED"} in {time.time()-t0:.1f}s')
    return 'ok' if ok else 'failed'

# for case in sorted(pathlib.Path('/content/cases').iterdir()):
#     run_case(case)
print(f'{len(done)} cases already complete')